In [0]:
%sql
SELECT * FROM `onesource_eu_dev_rni`.`ux_sn_global`.`ux_sn_conso_master_table`;

## Shallow Clone vs Deep Clone vs CTAS 

Here is a summary table comparing **shallow clone**, **deep clone**, and **CTAS (Create Table As Select)**, including their differences and how to achieve each using **SQL**, **PySpark**, and **SQL + PySpark**.

### Comparison Table

| Feature              | Shallow Clone                                 | Deep Clone                                    | CTAS (Create Table As Select)             |
| -------------------- | --------------------------------------------- | --------------------------------------------- | ----------------------------------------- |
| **Data Copy**        | No (references source data files)             | Yes (copies data files to new location)       | Yes (writes new data files)               |
| **Metadata Copy**    | Yes (schema, partitioning, invariants, nulls) | Yes (schema, partitioning, invariants, nulls) | Yes (schema only)                         |
| **Delta History**    | No (starts new history)                       | No (starts new history)                       | No (starts new history)                   |
| **Table Properties** | Most, but not all                             | Most, but not all                             | No (must specify manually if needed)      |
| **Space Usage**      | Minimal (until modified)                      | Full (data duplicated)                        | Full (data duplicated)                    |
| **Performance**      | Fastest                                       | Slower (copies data)                          | Slower (copies data)                      |
| **Isolation**        | Yes (modifications are independent)           | Yes (modifications are independent)           | Yes (modifications are independent)       |
| **Use Case**         | Testing, dev, experimentation                 | Backup, migration, archiving                  | Data transformation, subset, or new table |

***

## How to Achieve Each

| Approach          | Shallow Clone                                                      | Deep Clone                                                          | CTAS                                                               |
| ----------------- | ------------------------------------------------------------------ | ------------------------------------------------------------------- | ------------------------------------------------------------------ |
| **SQL**           | `CREATE TABLE ... SHALLOW CLONE source_table`                      | `CREATE TABLE ... DEEP CLONE source_table`                          | `CREATE TABLE ... AS SELECT * FROM source_table`                   |
| **PySpark**       | `DeltaTable.forName(spark, ...).clone(target=..., isShallow=True)` | `DeltaTable.forName(spark, ...).clone(target=..., isShallow=False)` | `df = spark.table(...); df.write.format("delta").saveAsTable(...)` |
| **SQL + PySpark** | `spark.sql("CREATE TABLE ... SHALLOW CLONE ...")`                  | `spark.sql("CREATE TABLE ... DEEP CLONE ...")`                      | `spark.sql("CREATE TABLE ... AS SELECT * FROM ...")`               |

***

## Quick Notes

*   **Shallow clone:** Fast, space-efficient, no data duplication, but depends on source data files.
*   **Deep clone:** Full, independent copy of data and metadata, but no Delta history.
*   **CTAS:** Copies data and schema only, not full Delta metadata or properties.

You can use **SQL directly** in a notebook cell, **PySpark** with the `DeltaTable` API, or **combine both** by running SQL statements via `spark.sql()` in PySpark.


In [0]:
%sql
CREATE OR REPLACE TABLE onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table_dbapp
DEEP CLONE onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table;

In [0]:
%sql
SELECT * FROM onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table_dbapp;


In [0]:
%python
_sqldf.printSchema()

root
 |-- match_type: string (nullable = true)
 |-- sp_test_id: string (nullable = true)
 |-- cl_test_id: string (nullable = true)
 |-- brand_L1: string (nullable = true)
 |-- branded_flag: string (nullable = true)
 |-- flavour_pack: string (nullable = true)
 |-- prod_type: string (nullable = true)
 |-- test_country: string (nullable = true)
 |-- test_year: double (nullable = true)
 |-- zone: string (nullable = true)
 |-- activated: double (nullable = true)
 |-- competitor1: string (nullable = true)
 |-- competitor2: string (nullable = true)
 |-- pack: string (nullable = true)
 |-- prod_catL4: string (nullable = true)
 |-- prod_name_pack: string (nullable = true)
 |-- stage: string (nullable = true)
 |-- usage: string (nullable = true)
 |-- CBU: string (nullable = true)
 |-- EUtop100: string (nullable = true)
 |-- NS_global_range: double (nullable = true)
 |-- act_std_reached: string (nullable = true)
 |-- brand_L0: string (nullable = true)
 |-- claims_number: double (nullable = true)


In [0]:
# Get the schema of the original table to build CREATE TABLE statement
original_df = spark.table("onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table")
schema_fields = original_df.schema

# Build column definitions for CREATE TABLE
column_defs = []
for field in schema_fields.fields:
    nullable = "" if field.nullable else "NOT NULL"
    column_defs.append(f"{field.name} {field.dataType.simpleString()} {nullable}")

columns_ddl = ",\n  ".join(column_defs)

# Drop table if exists
spark.sql("DROP TABLE IF EXISTS onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table_dbapp")

# Create table with IDENTITY column using SQL
spark.sql(f"""
CREATE TABLE onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table_dbapp (
  row_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1),
  {columns_ddl}
)
""")

# Insert data from original table
spark.sql("""
INSERT INTO onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table_dbapp 
SELECT * FROM onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table
""")

---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
File <command-7408747381959006>, line 1
----> 1 from delta.tables import DeltaTable, IdentityGenerator
      2 from pyspark.sql.types import LongType
      4 # Get the schema of the original table

ImportError: cannot import name 'IdentityGenerator' from 'delta.tables' (/databricks/python/lib/python3.10/site-packages/delta/tables.py)

In [0]:
%sql
DROP TABLE IF EXISTS onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table_dbapp;

In [0]:
%sql
CREATE TABLE onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table_dbapp (
  row_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1),
  match_type STRING,
  sp_test_id STRING,
  cl_test_id STRING,
  brand_L1 STRING,
  branded_flag STRING,
  flavour_pack STRING,
  prod_type STRING,
  test_country STRING,
  test_year DOUBLE,
  zone STRING,
  activated DOUBLE,
  competitor1 STRING,
  competitor2 STRING,
  pack STRING,
  prod_catL4 STRING,
  prod_name_pack STRING,
  stage STRING,
  usage STRING,
  CBU STRING,
  EUtop100 STRING,
  NS_global_range DOUBLE,
  act_std_reached STRING,
  brand_L0 STRING,
  claims_number DOUBLE,
  comment STRING,
  competitor1_size STRING,
  competitor2_size STRING,
  dashboard_usage STRING,
  cl_delivered DOUBLE,
  hero_flag STRING,
  prod_catL1 STRING,
  prod_catL2 DOUBLE,
  prod_catL3 DOUBLE,
  prod_status STRING,
  proxy2_catL4 STRING,
  proxy3_catL4 STRING,
  proxy_catL4 STRING,
  retest_year STRING,
  sample_size STRING,
  score_main STRING,
  score_secondary STRING,
  test_area STRING,
  test_count DOUBLE,
  test_danoneprod STRING,
  test_environment STRING,
  test_status STRING,
  top30 STRING,
  uniquekey_combo STRING,
  cl_competitor1_pct DOUBLE,
  cl_competitor1_result STRING,
  cl_competitor2_pct DOUBLE,
  cl_competitor2_result STRING,
  cl_competitor3 STRING,
  cl_competitor3_pct DOUBLE,
  cl_competitor3_sup_result STRING,
  pack_size STRING,
  cl_pct STRING,
  cl_pyramid_level STRING,
  cl_questionnaire STRING,
  report_released STRING,
  cl_sku_id STRING,
  cl_study_name STRING,
  cl_substantiated STRING,
  cl_theme STRING,
  Unnamed__64 STRING,
  Unnamed__65 DOUBLE,
  Unnamed__66 DOUBLE,
  ingestion_timestamp TIMESTAMP
);

In [0]:
%sql
INSERT INTO onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table_dbapp (
  match_type, sp_test_id, cl_test_id, brand_L1, branded_flag, flavour_pack, prod_type, 
  test_country, test_year, zone, activated, competitor1, competitor2, pack, prod_catL4, 
  prod_name_pack, stage, usage, CBU, EUtop100, NS_global_range, act_std_reached, brand_L0, 
  claims_number, comment, competitor1_size, competitor2_size, dashboard_usage, cl_delivered, 
  hero_flag, prod_catL1, prod_catL2, prod_catL3, prod_status, proxy2_catL4, proxy3_catL4, 
  proxy_catL4, retest_year, sample_size, score_main, score_secondary, test_area, test_count, 
  test_danoneprod, test_environment, test_status, top30, uniquekey_combo, cl_competitor1_pct, 
  cl_competitor1_result, cl_competitor2_pct, cl_competitor2_result, cl_competitor3, 
  cl_competitor3_pct, cl_competitor3_sup_result, pack_size, cl_pct, cl_pyramid_level, 
  cl_questionnaire, report_released, cl_sku_id, cl_study_name, cl_substantiated, cl_theme, 
  Unnamed__64, Unnamed__65, Unnamed__66, ingestion_timestamp
)
SELECT * FROM onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table;

In [0]:
%sql
ALTER TABLE onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table_dbapp
ADD CONSTRAINT pk_row_id PRIMARY KEY(row_id);

In [0]:
%sql
SELECT row_id, match_type, sp_test_id, cl_test_id, brand_L1 
FROM onesource_eu_dev_rni.ux_sn_global.ux_sn_conso_master_table_dbapp 
LIMIT 10;

In [ ]:
CREATE TABLE IF NOT EXISTS dataintel_usecase.ux_sn_global.ux_sn_conso_master_table_dbapp_audit (
  audit_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1),
  event_ts TIMESTAMP,
  event_type STRING,              -- EDIT_START / UPDATE_CELL / INSERT_ROW / DELETE_ROW / SAVE
  user_name STRING,               -- from headers or fallback
  session_id STRING,              -- per browser session
  page_no INT,
  table_fqn STRING,
  row_id BIGINT,
  col_name STRING,
  old_value STRING,
  new_value STRING,
  notes STRING
)
USING DELTA;